In [21]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper, PDFTableExtractor

helper = Helper()

In [2]:
import fitz
import pandas as pd


# =========================
# 1. ROW EXTRACTION (Y-axis)
# =========================
def extract_rows(page, bbox=None):
    items = []

    for block in page.get_text("dict")["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            for span in line["spans"]:
                x0, y0, x1, y1 = span["bbox"]
                text = span["text"].strip()

                if not text:
                    continue

                if bbox:
                    bx0, by0, bx1, by1 = bbox
                    if not (x0 >= bx0 and y0 >= by0 and x1 <= bx1 and y1 <= by1):
                        continue

                items.append({
                    "x0": x0, "y0": y0,
                    "x1": x1, "y1": y1,
                    "text": text,
                    "x_center": (x0 + x1) / 2,
                    "y_center": (y0 + y1) / 2,
                    "height": y1 - y0
                })

    if not items:
        return []

    items.sort(key=lambda x: x["y_center"])

    heights = [i["height"] for i in items]
    avg_height = sum(heights) / len(heights)
    y_threshold = avg_height * 0.5

    rows = []
    current_row = [items[0]]

    for i in range(1, len(items)):
        if abs(items[i]["y_center"] - current_row[-1]["y_center"]) < y_threshold:
            current_row.append(items[i])
        else:
            rows.append(current_row)
            current_row = [items[i]]

    rows.append(current_row)

    return rows


# =========================
# 2. ANCHOR CUT (ROW LEVEL)
# =========================
def apply_anchor_cut_rows(rows, top_left=None, top_right=None):
    new_rows = []

    for row in rows:
        filtered = []

        for item in row:
            keep = True

            if top_left:
                ax, ay = top_left
                if item["y1"] < ay or item["x1"] < ax:
                    keep = False

            if top_right:
                rx = top_right[0]
                if item["x0"] > rx:
                    keep = False

            if keep:
                filtered.append(item)

        if filtered:
            new_rows.append(filtered)

    return new_rows


# =========================
# 3. COLUMN ASSIGN (X-axis)
# =========================
def assign_columns(rows, x_lines, tol=10):
    table = []

    for row in rows:
        cols = [[] for _ in range(len(x_lines))]

        for item in row:
            assigned = False

            # pass-through (strong signal)
            for idx, lx in enumerate(x_lines):
                if item["x0"] - tol <= lx <= item["x1"] + tol:
                    cols[idx].append(item)
                    assigned = True
                    break

            # proximity fallback
            if not assigned:
                dists = [abs(item["x_center"] - lx) for lx in x_lines]
                idx = dists.index(min(dists))
                cols[idx].append(item)

        # join text
        row_text = []
        for col in cols:
            col_sorted = sorted(col, key=lambda x: x["x0"])
            row_text.append(" ".join(i["text"] for i in col_sorted))

        table.append(row_text)

    return table


# =========================
# 4. MAIN PIPELINE
# =========================
def extract_table(
    pdf_path,
    page_no=0,
    bbox=None,
    x_lines=None,
    top_left=None,
    top_right=None
):
    if not x_lines:
        raise ValueError("x_lines required")

    doc = fitz.open(pdf_path)
    page = doc[page_no]

    # 1. rows
    rows = extract_rows(page, bbox)

    # 2. anchor cut
    rows = apply_anchor_cut_rows(rows, top_left, top_right)

    if not rows:
        return pd.DataFrame()

    # 3. columns
    table = assign_columns(rows, x_lines)

    df = pd.DataFrame(table)

    doc.close()
    return df


# =========================
# 5. SAVE
# =========================
def save_to_excel(df, path):
    df.to_excel(path, index=False)

In [3]:
from app.utils import Helper
import subprocess

helper = Helper()

pdf_path = "SAM.pdf"

masked_pdf = helper.mask_outside_bboxes(
    pdf_path,
    [(183.46, 37.68, 430.81, 787.9)]
)

subprocess.Popen([masked_pdf], shell=True)

df = extract_table(
    pdf_path=masked_pdf,   # 🔥 use masked pdf
    page_no=1,
    bbox=None,             # already masked
    x_lines=[220, 415],    # from your GUI tool
    # top_left=(430, 120),   # optional
    # top_right=(610, 120)   # optional
)

# print(df)

save_to_excel(df, "soutput.xlsx")

In [33]:
#29-04-2026 CODE

import re
import random
import pandas as pd
import fitz

def extract_rows_sampling(page, bbox=None):
    """
    Sampling-based row detection
    Returns DataFrame with 1 column (each row = full text)
    """

    # --- get spans ---
    items = []

    for block in page.get_text("dict")["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            for span in line["spans"]:
                x0, y0, x1, y1 = span["bbox"]
                text = span["text"].strip()

                if not text:
                    continue

                if bbox:
                    bx0, by0, bx1, by1 = bbox
                    if not (x0 >= bx0 and y0 >= by0 and x1 <= bx1 and y1 <= by1):
                        continue

                items.append({
                    "x0": x0, "y0": y0,
                    "x1": x1, "y1": y1,
                    "text": text,
                    "x_center": (x0 + x1) / 2,
                    "y_center": (y0 + y1) / 2,
                    "height": y1 - y0
                })

    if not items:
        return pd.DataFrame()

    # --- STEP 1: sample vertical lines ---
    page_width = page.rect.width
    y_hits = []

    for _ in range(60):  # slightly more stable than 50
        x = random.uniform(0, page_width)

        for item in items:
            if item["x0"] <= x <= item["x1"]:
                y_hits.append(item["y_center"])

    if not y_hits:
        return pd.DataFrame()

    # --- STEP 2: cluster y_hits into row anchors ---
    y_hits.sort()

    heights = [i["height"] for i in items]
    avg_height = sum(heights) / len(heights)
    threshold = avg_height * 0.6   # adaptive

    rows_y = []
    current = [y_hits[0]]

    for i in range(1, len(y_hits)):
        if abs(y_hits[i] - current[-1]) < threshold:
            current.append(y_hits[i])
        else:
            rows_y.append(sum(current) / len(current))
            current = [y_hits[i]]

    rows_y.append(sum(current) / len(current))

    # --- STEP 3: assign spans to nearest row ---
    rows = [[] for _ in rows_y]

    for item in items:
        distances = [abs(item["y_center"] - y) for y in rows_y]
        idx = distances.index(min(distances))
        rows[idx].append(item)

    # --- STEP 4: convert rows → text ---
    table = []

    for row in rows:
        row_sorted = sorted(row, key=lambda x: x["x0"])
        text = " ".join(i["text"] for i in row_sorted)
        table.append([text])
        table.extend([""]*4)

    # --- return as single column df ---
    # df = pd.DataFrame(table, columns=["row_text"])
    # return df
    return rows

def assign_columns_from_rows(rows, x_lines, tol=10):
    """
    rows: [[item, item], ...]
    x_lines: [x1, x2, ...]
    """

    table = []

    for row in rows:
        cols = [[] for _ in range(len(x_lines))]

        for item in row:
            assigned = False

            # --- pass-through check ---
            for idx, lx in enumerate(x_lines):
                if item["x0"] - tol <= lx <= item["x1"] + tol:
                    cols[idx].append(item)
                    assigned = True
                    break

            # --- proximity fallback ---
            if not assigned:
                dists = [abs(item["x_center"] - lx) for lx in x_lines]
                idx = dists.index(min(dists))
                cols[idx].append(item)

        # --- join text per column ---
        row_text = []
        for col in cols:
            col_sorted = sorted(col, key=lambda x: x["x0"])
            text = " ".join(i["text"] for i in col_sorted)
            row_text.append(text)

        table.append(row_text)

    return pd.DataFrame(table)

def find_anchor_y(page, keyword, bbox=None):
    def norm(s):
        return re.sub(r"\s+", " ", s).strip().lower()

    keyword = norm(keyword)

    for block in page.get_text("dict")["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            x0, y0, x1, y1 = line["bbox"]

            # 🔥 bbox restriction (important for multi-table pages)
            if bbox:
                bx0, by0, bx1, by1 = bbox
                if not (x0 >= bx0 and y0 >= by0 and x1 <= bx1 and y1 <= by1):
                    continue

            line_text = " ".join(span["text"] for span in line["spans"])

            if keyword in norm(line_text):
                return y0   # top of anchor line

    return None

def cut_rows_above_anchor(rows, anchor_y):
    if anchor_y is None:
        return rows

    new_rows = []

    for row in rows:
        # keep row if ANY item is below anchor
        if any(item["y1"] >= anchor_y for item in row):
            new_rows.append(row)

    return new_rows

In [43]:
pth = r"C:\Users\rando\Office Projects\rep_fsparse\pdf_url.json"
configs = helper.load_json(pth)
# configs 

In [ ]:
import fitz
import pandas as pd

cfg = configs["hdfc_2"]
path = cfg["path"]
print(path)
doc = fitz.open(path)

all_dfs = []

for page_no in range(doc.page_count):
    page = doc[page_no]

    for table in cfg["tables"]:

        bbox = tuple(table["bbox"]) if table.get("bbox") else None
        x_lines = table.get("x_lines", [])
        anchor = table.get("anchor", "")

        # --- skip incomplete config ---
        if not bbox or not x_lines:
            continue

        # 1. rows
        rows = extract_rows_sampling(page, bbox)
        if not rows:
            continue

        # 2. anchor (optional)
        if anchor:
            anchor_y = find_anchor_y(page, anchor, bbox)
            rows = cut_rows_above_anchor(rows, anchor_y)

        # 3. columns
        df = assign_columns_from_rows(rows, x_lines)

        # 4. metadata
        df["page"] = page_no
        df["table"] = table.get("name", "unknown")

        all_dfs.append(df)

doc.close()

# --- safe concat ---
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()
    
final_df.to_csv(path.replace(".pdf",".xlsx"))

HDF2.pdf


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [20]:
path = r"GEN.pdf"
path = r"ADI.pdf"
path = r"ABI.pdf"
doc = fitz.open(path)

bbox = (183.46, 4.1, 430.81, 791.18) #generali
bbox = (13.92, 101.56, 306.32, 841.96) #adi
bboxes = [(1.74, 184.6, 210.72, 644.34),(195.04, 183.72, 397.05, 640.86),(390.96, 182.85, 591.23, 584.26)]
all_dfs = []

x_lines = [428.4, 560.75]   #adi
keyword = ""

for page_no in range(doc.page_count):
    page = doc[page_no]

    for i, bbox in enumerate(bboxes):

        # 1. rows (bbox scoped)
        rows = extract_rows_sampling(page, bbox)

        if not rows:
            continue
        
        if keyword:
            anchor_y = find_anchor_y(page, keyword)

            # 3. cut (IMPORTANT: bbox-aware)
            rows = cut_rows_above_anchor(rows, anchor_y)

        # 4. columns
        df = assign_columns_from_rows(rows, x_lines)

        # tagging (very useful later)
        df["page"] = page_no + 1
        df["region"] = i

        all_dfs.append(df)

doc.close()

final_df = pd.concat(all_dfs, ignore_index=True)
final_df.to_excel(path.replace(".pdf",".xlsx"))